# Bit Manipulation


## Topic overview

Operate directly on binary representations.

## Pattern-recognition rules

- XOR to cancel pairs.
- Population count.
- Bitmask DP.

## Common data structures

- `int` (arbitrary precision in Python)

## Standard complexity expectations

- Often O(n) with O(1) space via masks.

## Common mistakes

- Confusing `&` with `and`.
- Sign-extension quirks in non-Python languages.

## Original illustrative example

In [ ]:
# Replace with an ORIGINAL example. Do not paste external
# problem statements. See src/algorithms/ for reusable helpers.
example_input = []
example_expected = None

## Add solved problems below

Each new sub-section should follow the template in `../templates/notebook_template.ipynb`.

1. [Divide Two Integers](#divide-two-integers)


# Divide Two Integers

## Metadata

- Source: LeetCode 29
- Problem URL: https://leetcode.com/problems/divide-two-integers/
- Difficulty: Medium
- Topic: Bit Manipulation
- Date started: 2026-09-06
- Date solved: 2026-09-06
- Current mastery level: 1
- Last reviewed: 2026-09-06
- Next review:


## Problem statement in my own words

Compute `dividend / divisor` as an integer, **without** using
multiplication, division, or modulo (`*`, `/`, `//`, `%`).

The result must truncate toward zero: drop the fractional part. So
$10 / 3 \rightarrow 3$ and $7 / -3 \rightarrow -2$, not $-3$.

The environment is 32-bit signed:

$$
[-2^{31},\ 2^{31}-1] = [-2147483648,\ 2147483647]
$$

If the true quotient would fall outside that range, clamp it. The only
case that overflows is

$$
(-2^{31}) / (-1) = 2^{31}
$$

which must return $2^{31}-1$.

`divisor` is never zero.


## Inputs, outputs, and constraints

- Inputs: `dividend` (int), `divisor` (int, nonzero)
- Outputs: `int` — the truncated 32-bit quotient
- Constraints:
  - $-2^{31} \le$ `dividend`, `divisor` $\le 2^{31}-1$
  - `divisor != 0`

Allowed operations: addition, subtraction, bit shifts, comparisons.
Forbidden: `*`, `/`, `//`, `%`.


## Examples

| Input | Expected | Notes |
|---|---|---|
| `dividend = 10`, `divisor = 3` | `3` | $3.333\ldots$ truncated toward zero |
| `dividend = 7`, `divisor = -3` | `-2` | $-2.333\ldots$ truncated toward zero, **not** floored to $-3$ |
| `dividend = -2147483648`, `divisor = -1` | `2147483647` | only overflow case; clamp to `INT_MAX` |

Python's `//` floors toward $-\infty$, so `7 // -3` is `-3`. That is
**not** this problem. Do not verify against `//` on negative inputs.


## Initial observations

- Repeated subtraction works: take `abs(divisor)` out of `abs(dividend)`
  until it no longer fits. Too slow when the quotient is near $2^{31}$.
- Left shift replaces doubling: $x \ll k = x \cdot 2^k$ without using
  `*`.
- Every nonnegative quotient is a sum of powers of two. Building it from
  bit $31$ down to bit $0$ is binary decomposition, not trial
  subtraction of $1$ copy at a time.
- Sign is XOR of the input signs: the result is negative iff exactly one
  of `dividend` and `divisor` is negative.
- Work with `a = abs(dividend)` and `b = abs(divisor)`, then restore the
  sign. Python's `abs(-2**31)` is safe; in C++/Java it overflows.


## Brute-force approach

### Why it works

Count how many times `b` fits into `a` by subtracting `b` once per loop.
Correct, and it never uses `*`, `/`, or `%`. If the quotient is $Q$,
the loop runs $Q$ times — up to about $2^{31}$ iterations.

### Implementation


In [ ]:
INT_MIN = -(1 << 31)
INT_MAX = (1 << 31) - 1


def brute_force(dividend: int, divisor: int) -> int:
    # The only 32-bit overflow: INT_MIN / -1 = 2^31.
    if dividend == INT_MIN and divisor == -1:
        return INT_MAX

    negative = (dividend < 0) != (divisor < 0)
    a = abs(dividend)
    b = abs(divisor)

    quotient = 0
    while a >= b:
        a -= b
        quotient += 1

    if negative:
        quotient = -quotient
    return quotient


### Complexity

- Time: $O(|Q|)$ where $Q$ is the quotient — too slow near $2^{31}$
- Space: $O(1)$


## Optimized insight

Instead of subtracting one copy of $b$ at a time, subtract the largest
power-of-two multiple that still fits.

For $10 / 3$:

$$
3 \ll 0 = 3,\qquad 3 \ll 1 = 6,\qquad 3 \ll 2 = 12
$$

$12$ overshoots. $6 \le 10$, so two copies fit:

$$
10 - 6 = 4,\qquad \text{quotient bit } 1 \text{ turns on } (+2)
$$

Then $3 \le 4$, so one more copy fits:

$$
4 - 3 = 1,\qquad \text{quotient bit } 0 \text{ turns on } (+1)
$$

Remainder $1$ is dropped (truncate toward zero). Quotient $= 2+1 = 3$.

That is the binary expansion of $3 = 2^1 + 2^0$. The loop is writing
the quotient's bits from most significant to least significant.

## Optimized approach

1. Record `negative = (dividend < 0) != (divisor < 0)`.
2. Set `a = abs(dividend)`, `b = abs(divisor)`, `quotient = 0`.
3. For `shift` from $31$ down to $0$:
   - if `(b << shift) <= a`:
     - `a -= b << shift`
     - `quotient += 1 << shift`  (turn on bit `shift`)
4. If `negative`, negate `quotient`.
5. Clamp into `[INT_MIN, INT_MAX]` and return.

### Step-by-step trace — Example 1

`dividend = 10`, `divisor = 3`. Both positive, so `a = 10`, `b = 3`.

Shifts $31 \ldots 2$ fail because $3 \ll 2 = 12 > 10$.

| shift | $b \ll$ shift | fits? | $a$ after | quotient bits | quotient |
|---|---|---|---|---|---|
| 2 | 12 | no | 10 | `000` | 0 |
| 1 | 6 | yes | 4 | `010` | 2 |
| 0 | 3 | yes | 1 | `011` | 3 |

Answer: $3$.

### Trace — Example 2

`dividend = 7`, `divisor = -3`. Exactly one sign is negative.

`a = 7`, `b = 3`. Same magnitude walk: $6 \le 7$ → quotient $2$,
remainder $1$. Then restore the sign: $-2$.

### Overflow

`dividend = -2147483648`, `divisor = -1` produces $+2^{31}$. The final
clamp returns `INT_MAX = 2147483647`.


In [ ]:
class Solution:
    def divide(self, dividend: int, divisor: int) -> int:
        INT_MIN = -(1 << 31)
        INT_MAX = (1 << 31) - 1

        # Negative iff exactly one input is negative (XOR on signs).
        negative = (dividend < 0) != (divisor < 0)

        # Magnitudes only. Sign is restored after the bit walk.
        a = abs(dividend)
        b = abs(divisor)

        quotient = 0

        # Turn on quotient bits from MSB to LSB.
        for shift in range(31, -1, -1):
            if (b << shift) <= a:
                a -= b << shift
                quotient += 1 << shift

        if negative:
            quotient = -quotient

        if quotient > INT_MAX:
            return INT_MAX
        if quotient < INT_MIN:
            return INT_MIN
        return quotient


def optimized(dividend: int, divisor: int) -> int:
    return Solution().divide(dividend, divisor)


## Complexity analysis

The loop inspects 32 bit positions. Each iteration is $O(1)$ arithmetic
(Python big-ints, but the values stay near 32 bits).

- Time: $O(32) = O(1)$ for 32-bit inputs; $O(\log |\text{dividend}|)$
  in general
- Space: $O(1)$

Repeated subtraction is $O(|Q|)$. The shift version replaces that linear
scan with one pass over the bits of $Q$.


## Edge cases

- `INT_MIN / -1` → clamp to `INT_MAX`
- `INT_MIN / 1` → `INT_MIN`
- `0 / x` → `0`
- signs: `(+, -)` and `(-, +)` truncate toward zero, not toward
  $-\infty$
- `divisor = ±1` — still correct; the bit walk just copies `abs(dividend)`
- equal magnitudes → `±1`


## Testing


In [ ]:
assert optimized(10, 3) == 3
assert optimized(7, -3) == -2
assert optimized(-7, 3) == -2
assert optimized(-7, -3) == 2
assert optimized(1, 1) == 1
assert optimized(0, 1) == 0
assert optimized(INT_MIN, -1) == INT_MAX
assert optimized(INT_MIN, 1) == INT_MIN
assert optimized(INT_MAX, 1) == INT_MAX
assert brute_force(10, 3) == 3
assert brute_force(7, -3) == -2
assert brute_force(-7, 3) == -2
assert brute_force(0, 1) == 0

print("Divide Two Integers checks passed")


## Visualization

Each frame is one bit of the quotient, from a few bits above the first
hit down to bit $0$. The left panel compares the remaining dividend to
$b \ll \text{shift}$. The right panel lights up the bits that have
already been written.

If `ipywidgets` is installed, use **Next step** / **Prev** / **Reset**.
Otherwise every recorded bit is drawn as a static figure.


In [ ]:
from dataclasses import dataclass
from typing import List, Optional

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle


@dataclass(frozen=True)
class DivideFrame:
    shift: int
    shifted: int
    remaining_before: int
    fits: bool
    remaining_after: int
    quotient: int
    decision: str


def divide_frames(dividend: int, divisor: int) -> List[DivideFrame]:
    """Record the bit walk exactly as Solution.divide executes."""
    a = abs(dividend)
    b = abs(divisor)
    quotient = 0

    highest = 0
    for shift in range(31, -1, -1):
        if (b << shift) <= a:
            highest = shift
            break
    start = min(31, highest + 1)

    frames: List[DivideFrame] = []
    for shift in range(start, -1, -1):
        shifted = b << shift
        before = a
        if shifted <= a:
            a -= shifted
            quotient += 1 << shift
            decision = f"fits → a -= {shifted}, quotient += {1 << shift}"
            frames.append(
                DivideFrame(shift, shifted, before, True, a, quotient, decision)
            )
        else:
            frames.append(
                DivideFrame(
                    shift,
                    shifted,
                    before,
                    False,
                    a,
                    quotient,
                    f"{shifted} > {before} → skip bit {shift}",
                )
            )
    return frames


def plot_divide_frame(
    dividend: int,
    divisor: int,
    frame: DivideFrame,
    *,
    bit_hi: Optional[int] = None,
    ax_bar=None,
    ax_bits=None,
):
    """Two-panel snapshot of one quotient bit decision."""
    created_fig = ax_bar is None
    if created_fig:
        _, (ax_bar, ax_bits) = plt.subplots(
            1, 2, figsize=(12, 4.0), gridspec_kw={"width_ratios": [1.25, 1.1]}
        )

    start_a = abs(dividend)
    scale = max(start_a, frame.shifted, 1)
    ax_bar.set_xlim(0, scale * 1.18)
    ax_bar.set_ylim(-0.7, 2.4)
    ax_bar.set_yticks([0.4, 1.5])
    ax_bar.set_yticklabels(["candidate", "remaining"])

    remain_color = "#2a9d8f" if frame.fits else "#4c78a8"
    ax_bar.add_patch(
        Rectangle((0, 1.15), frame.remaining_before, 0.7, color=remain_color, alpha=0.9)
    )
    cand_color = "#2a9d8f" if frame.fits else "#e76f51"
    ax_bar.add_patch(
        Rectangle((0, 0.05), frame.shifted, 0.7, color=cand_color, alpha=0.85)
    )
    ax_bar.text(
        frame.remaining_before + scale * 0.02,
        1.5,
        str(frame.remaining_before),
        va="center",
        fontsize=10,
    )
    ax_bar.text(
        frame.shifted + scale * 0.02,
        0.4,
        f"{frame.shifted}  (= {abs(divisor)} << {frame.shift})",
        va="center",
        fontsize=10,
    )
    ax_bar.set_title(
        f"{dividend} ÷ {divisor}   shift={frame.shift}   "
        f"{'TAKE' if frame.fits else 'SKIP'}\n{frame.decision}",
        loc="left",
    )
    ax_bar.set_xlabel("magnitude")
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)

    if bit_hi is None:
        bit_hi = max(frame.shift, 3)
    bits = list(range(bit_hi, -1, -1))
    for i, bit in enumerate(bits):
        on = (frame.quotient >> bit) & 1
        is_current = bit == frame.shift
        face = "#2a9d8f" if on else "#edf2f4"
        edge = "#9b2226" if is_current else "#1d3557"
        lw = 2.6 if is_current else 1.0
        ax_bits.add_patch(
            FancyBboxPatch(
                (i, 0.25),
                0.85,
                1.15,
                boxstyle="round,pad=0.04,rounding_size=0.12",
                facecolor=face,
                edgecolor=edge,
                linewidth=lw,
            )
        )
        ax_bits.text(
            i + 0.42,
            0.82,
            str(on),
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            color="#1d3557",
        )
        ax_bits.text(i + 0.42, 0.05, f"2^{bit}", ha="center", va="top", fontsize=8)
    ax_bits.set_xlim(-0.2, len(bits))
    ax_bits.set_ylim(-0.25, 2.15)
    ax_bits.axis("off")
    ax_bits.set_title(
        f"quotient bits = {frame.quotient}   remaining after = {frame.remaining_after}",
        loc="left",
    )

    if created_fig:
        plt.tight_layout()
    return ax_bar, ax_bits


def visualize_divide_all(dividend: int, divisor: int) -> None:
    frames = divide_frames(dividend, divisor)
    bit_hi = max(f.shift for f in frames)
    sign = "-" if (dividend < 0) != (divisor < 0) else "+"
    print(f"{dividend} / {divisor}  →  sign {sign}  answer {optimized(dividend, divisor)}")
    print(f"{'shift':>5}  {'b<<s':>6}  {'a':>6}  {'fit':>3}  {'q':>4}  decision")
    for frame in frames:
        print(
            f"{frame.shift:>5}  {frame.shifted:>6}  {frame.remaining_before:>6}  "
            f"{'Y' if frame.fits else 'n':>3}  {frame.quotient:>4}  {frame.decision}"
        )
        plot_divide_frame(dividend, divisor, frame, bit_hi=bit_hi)
    plt.show()


def step_through_divide(dividend: int = 10, divisor: int = 3) -> None:
    """Interactive Next-step walker; falls back to all frames."""
    frames = divide_frames(dividend, divisor)
    bit_hi = max(f.shift for f in frames)
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output, display
    except ImportError:
        visualize_divide_all(dividend, divisor)
        return

    output = widgets.Output()
    state = {"i": 0}

    def render() -> None:
        with output:
            clear_output(wait=True)
            frame = frames[state["i"]]
            print(
                f"{dividend} / {divisor}   "
                f"frame {state['i'] + 1}/{len(frames)}"
            )
            plot_divide_frame(dividend, divisor, frame, bit_hi=bit_hi)
            plt.show()

    def on_next(_=None) -> None:
        state["i"] = min(state["i"] + 1, len(frames) - 1)
        render()

    def on_prev(_=None) -> None:
        state["i"] = max(state["i"] - 1, 0)
        render()

    def on_reset(_=None) -> None:
        state["i"] = 0
        render()

    prev_btn = widgets.Button(description="Prev")
    next_btn = widgets.Button(description="Next step")
    reset_btn = widgets.Button(description="Reset")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    reset_btn.on_click(on_reset)
    display(widgets.HBox([prev_btn, next_btn, reset_btn]), output)
    render()


print("Example 1 — 10 / 3")
step_through_divide(10, 3)

print("\nExample 2 — 7 / -3")
step_through_divide(7, -3)


## Alternative approaches

- Repeated subtraction: correct, $O(|Q|)$, fails the time limit.
- Double the divisor in an inner loop (`cur = b; multiple = 1; while
  cur + cur <= a: cur += cur; multiple += multiple`) then subtract.
  Same idea, builds one exponential chunk at a time instead of testing
  every bit. Still $O(\log |Q|)$.
- Recursion on the same doubling loop. Same bound, extra stack.

## Mistakes I made

- Using Python `//` as an oracle on negatives — it floors, this problem
  truncates toward zero.
- Forgetting the `INT_MIN / -1` clamp.
- Shifting `b` so far that a 32-bit language would overflow. Python is
  safe; in C++/Java, compare before shifting or use 64-bit temps.
- Restoring the sign before clamping — clamp the signed result.

## Pattern recognition

Need multiplication or division by a power of two → shift.

```text
x << 1      double x
x << k      x * 2^k
1 << k      2^k
```

Need an integer quotient without `/` → write the quotient's bits from
MSB to LSB by testing `divisor << shift`.

## Related problems

- Multiply two integers without `*` — same binary decomposition, add
  instead of subtract
- Pow(x, n) — binary exponentiation, same bit walk
- Sum of Two Integers — addition with XOR / carry, no `+`
- Koko Eating Bananas — also searches a numeric answer, but that one
  uses binary search, not bit construction

## Real-world or engineering connection

Hardware integer dividers and restoring-division circuits do this: at
each bit position they test whether the (shifted) divisor fits, then
write that bit of the quotient. Software big-integer libraries use the
same restore-or-skip loop.

## Final takeaways

Do not memorize the clamp. Keep this:

1. signs → XOR, then work with `abs`
2. for `shift` from 31 down to 0, if `b << shift` fits, subtract it and
   turn on that bit of the quotient
3. restore the sign
4. clamp `INT_MIN / -1`

The reusable idea is **binary decomposition of the quotient**, not
repeated subtraction.


## Reattempt log

| Date | Mastery | Notes |
|---|---|---|
| 2026-09-06 | 1 | First write-up: MSB-to-LSB shift subtraction + 32-bit clamp |
